# Ajnyana — Data Pipeline

Kumpulkan, bersihkan, dan merge semua sumber teks Bahasa Sunda.

**Sources:**
- Wikipedia Sunda (`su`) — ~62K artikel
- CC-100 Sundanese (`su`) — ~10M tokens
- OSCAR Sundanese — TBD

**Output:** `data/train.txt`, `data/val.txt`, `data/stats.json`

In [1]:
from datasets import load_dataset
import re
import json
import os
import unicodedata
from collections import Counter

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Libraries OK')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries OK


## 1. Cleaning Functions

In [2]:
def clean_text(text: str) -> str:
    if not text or not isinstance(text, str):
        return ''

    # Normalize unicode
    text = unicodedata.normalize('NFC', text)

    # Hapus HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Hapus karakter kontrol
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    return text


def is_valid(text: str, min_words: int = 20) -> bool:
    if not text:
        return False
    words = text.split()
    if len(words) < min_words:
        return False
    # Minimal 50% karakter alfabet (filter noise/kode)
    alpha = sum(c.isalpha() for c in text)
    if alpha / max(len(text), 1) < 0.5:
        return False
    return True


print('Cleaning functions OK')

Cleaning functions OK


## 2. Source 1 — Wikipedia Sunda

In [3]:
print('Loading Wikipedia Sunda...')
wiki = load_dataset('wikimedia/wikipedia', '20231101.su', split='train')
print(f'Raw: {len(wiki):,} artikel')

wiki_texts = []
for item in wiki:
    text = clean_text(item['text'])
    if is_valid(text, min_words=30):
        wiki_texts.append(text)

print(f'After cleaning: {len(wiki_texts):,} artikel ({len(wiki_texts)/len(wiki)*100:.1f}% kept)')
print(f'Total words: {sum(len(t.split()) for t in wiki_texts):,}')
print(f'Sample:\n{wiki_texts[0][:300]}\n...')

Loading Wikipedia Sunda...


Generating train split: 100%|██| 61555/61555 [00:00<00:00, 370478.81 examples/s]


Raw: 61,555 artikel
After cleaning: 30,105 artikel (48.9% kept)
Total words: 5,229,689
Sample:
Wiktionary (atawa, dina édisi Sunda katelah Wikamus) mangrupa proyék sadulur Wikipédia nu tujuanana pikeun nyusun hiji kamus wikiwiki nu bébas (kaasup tésaurus & léxikon) dina sadaya basa. Nuturkeun pamanggih Daniel Alston, Wiktionary mimiti disusun 12 Désémber 2002. 29 Maret 2004 dua Wiktionary mun
...


## 3. Source 2 — CC-100 Sunda

In [6]:
print('Loading CC-100 Sunda...')
cc100 = load_dataset('allenai/c4', 'su', split='train')
print(f'Raw: {len(cc100):,} dokumen')

cc_texts = []
for item in cc100:
    text = clean_text(item['text'])
    if is_valid(text, min_words=20):
        cc_texts.append(text)

print(f'After cleaning: {len(cc_texts):,} dokumen ({len(cc_texts)/len(cc100)*100:.1f}% kept)')
print(f'Total words: {sum(len(t.split()) for t in cc_texts):,}')
print(f'Sample:\n{cc_texts[0][:300]}\n...')

Loading CC-100 Sunda...


Generating train split: 280719 examples [00:02, 98885.50 examples/s] 
Generating validation split: 269 examples [00:00, 42369.89 examples/s]


Raw: 280,719 dokumen
After cleaning: 238,988 dokumen (85.1% kept)
Total words: 56,729,601
Sample:
abditrass aplikator Oktober 10, 2019 New Google SEO Bandung, Indonesia
Home » 2019 » ABDITRASS - JASA GEOLISTRIK PALEMBANG TERBAIK DAN BERPENGALAMAN
Posted by Abditrass.com on Kamis, 10 Oktober 2019
...


## 4. Source 3 — OSCAR Sunda

In [8]:
print('Loading OSCAR Sunda...')
try:
    oscar = load_dataset('oscar-corpus/OSCAR-2301', language='su', split='train', trust_remote_code=True)
    print(f'Raw: {len(oscar):,} dokumen')

    oscar_texts = []
    for item in oscar:
        text = clean_text(item.get('text', item.get('content', '')))
        if is_valid(text, min_words=20):
            oscar_texts.append(text)

    print(f'After cleaning: {len(oscar_texts):,} dokumen ({len(oscar_texts)/len(oscar)*100:.1f}% kept)')
    print(f'Total words: {sum(len(t.split()) for t in oscar_texts):,}')
except Exception as e:
    print(f'OSCAR tidak tersedia atau error: {e}')
    oscar_texts = []

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'oscar-corpus/OSCAR-2301' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading OSCAR Sunda...
OSCAR tidak tersedia atau error: Dataset 'oscar-corpus/OSCAR-2301' is a gated dataset on the Hub. Visit the dataset page at https://huggingface.co/datasets/oscar-corpus/OSCAR-2301 to ask for access.


## 5. Merge + Deduplicate

In [9]:
all_texts = wiki_texts + cc_texts + oscar_texts
print(f'Total sebelum dedup: {len(all_texts):,} dokumen')

# Exact dedup by first 100 chars
seen = set()
deduped = []
for text in all_texts:
    key = text[:100].strip().lower()
    if key not in seen:
        seen.add(key)
        deduped.append(text)

print(f'After dedup: {len(deduped):,} dokumen ({len(deduped)/len(all_texts)*100:.1f}% kept)')

total_words = sum(len(t.split()) for t in deduped)
total_chars = sum(len(t) for t in deduped)
print(f'Total words: {total_words:,}')
print(f'Total chars: {total_chars:,}')
print(f'Est. tokens (~4 chars/token): {total_chars//4:,}')

Total sebelum dedup: 269,093 dokumen
After dedup: 266,394 dokumen (99.0% kept)
Total words: 61,189,263
Total chars: 413,511,612
Est. tokens (~4 chars/token): 103,377,903


## 6. Train/Val Split + Save

In [10]:
import random
random.seed(42)
random.shuffle(deduped)

split = int(len(deduped) * 0.98)
train_docs = deduped[:split]
val_docs   = deduped[split:]

print(f'Train: {len(train_docs):,} docs')
print(f'Val:   {len(val_docs):,} docs')

# Save
with open('../data/processed/train.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(train_docs))

with open('../data/processed/val.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(val_docs))

# Stats
stats = {
    'total_docs': len(deduped),
    'train_docs': len(train_docs),
    'val_docs':   len(val_docs),
    'sources': {
        'wikipedia': len(wiki_texts),
        'cc100':     len(cc_texts),
        'oscar':     len(oscar_texts),
    },
    'total_words': total_words,
    'est_tokens':  total_chars // 4,
}

with open('../data/processed/stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('\nSaved!')
print(json.dumps(stats, indent=2))

Train: 261,066 docs
Val:   5,328 docs

Saved!
{
  "total_docs": 266394,
  "train_docs": 261066,
  "val_docs": 5328,
  "sources": {
    "wikipedia": 30105,
    "cc100": 238988,
    "oscar": 0
  },
  "total_words": 61189263,
  "est_tokens": 103377903
}


## 7. Quality Check

In [11]:
# Distribusi panjang dokumen
lengths = [len(t.split()) for t in deduped]
print(f'Doc length (words):')
print(f'  Min:    {min(lengths)}')
print(f'  Max:    {max(lengths)}')
print(f'  Median: {sorted(lengths)[len(lengths)//2]}')
print(f'  Mean:   {sum(lengths)//len(lengths)}')

# Sample 3 random docs
print('\n=== SAMPLE DOCS ===')
for i, doc in enumerate(random.sample(deduped, 3)):
    print(f'\n[{i+1}] ({len(doc.split())} words)')
    print(doc[:400])
    print('...')

Doc length (words):
  Min:    20
  Max:    34049
  Median: 66
  Mean:   229

=== SAMPLE DOCS ===

[1] (170 words)
Nu Lila Di Anti - kenging Neni Yuhaenah - Fiksimini Basa Sunda
Nu Lila Di Anti
Dipidangkeun dina lapak 21 Januari 2015 06:13:51
“Aah..., lila-lila mah kapincut, geura...!” keukeuh nu ngolo. Diri tetep teu kagoda.
Kiwari, mangsa batur geus boga loba, jorojoy..., kabita. Lian ti pulasna nu rupa-rupa, umur kembangna gé awét, bisa tepikeun ka tilu bulanan. Paingan nempo batur mun geus resep, jiga kagé
...

[2] (414 words)
Gothenburg - Wikipédia Sunda, énsiklopédi bébas
(dialihkeun ti Göteborg)
Pikeun kagunaan séjén, tempo Gothenburg (disambiguasi).
Lokasi Gothenburg di Éropa Kalér
Koordinat: 57°42′N 11°58′E
County Västra Götaland
Piagem
450 km² (174 sq mi)
14,5 km² (5,6 sq mi) 3.2%
199 km² (77 sq mi)
3.717 km² (1.435 sq mi)
1.083/km² (2.807/sq mi)
495.849
- Kapadetan Urban
2.491/km² (6.439/sq mi)
- Kapadetan Metro

...

[3] (158 words)
Eunji | apink-topia
WordPress.com Tag Arch

## Next

Setelah pipeline selesai → `02_tokenizer.ipynb` — train BPE tokenizer dari corpus ini.